# Notebook 2: EGARCH & GJR-GARCH — Asymmetric Volatility
**Project:** GARCH & Volatility Forecasting
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from arch import arch_model
import statsmodels.api as sm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy = returns['SPY'] * 100
spy_raw = returns['SPY']
print(f'Loaded {len(spy):,} obs | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 obs | 2015-01-01 to 2024-12-31


## 1. Fit All Models

In [2]:
from garch_models import fit_garch, fit_egarch, fit_gjr_garch, compare_models
garch_r  = fit_garch(spy_raw)
egarch_r = fit_egarch(spy_raw)
gjr_r    = fit_gjr_garch(spy_raw)
comp = compare_models(spy_raw)
print('Model Comparison (sorted by AIC):')
print(comp)
comp.to_csv('../results/model_comparison_aic_bic.csv')

Model Comparison (sorted by AIC):
                AIC      BIC
model                       
GARCH(1,1)  7692.88  7716.34
GJR-GARCH   7694.22  7723.55
EGARCH      7729.73  7753.20


## 2. Conditional Volatility Comparison

In [3]:
fig, axes = plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
ax1.plot(garch_r['cond_vol'].index, garch_r['cond_vol']*100,  color='#185FA5',linewidth=1,alpha=0.85,label='GARCH(1,1)')
ax1.plot(egarch_r['cond_vol'].index,egarch_r['cond_vol']*100, color='#E24B4A',linewidth=1,alpha=0.85,label='EGARCH')
ax1.plot(gjr_r['cond_vol'].index,   gjr_r['cond_vol']*100,    color='#1D9E75',linewidth=1,alpha=0.85,label='GJR-GARCH')
ax1.set_title('Conditional Volatility: GARCH vs EGARCH vs GJR-GARCH'); ax1.legend(fontsize=9)
ax2=axes[1]
diff=(egarch_r['cond_vol']-garch_r['cond_vol'])*100
ax2.fill_between(diff.index,diff,0,where=diff>0,color='#E24B4A',alpha=0.6,label='EGARCH > GARCH')
ax2.fill_between(diff.index,diff,0,where=diff<0,color='#185FA5',alpha=0.4,label='EGARCH < GARCH')
ax2.set_title('EGARCH minus GARCH — Leverage Effect'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/02_egarch_gjr_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

## 3. News Impact Curves

In [4]:
z=np.linspace(-4,4,200)
gp=garch_r['params']; ep=garch_r['params']
omega_g=garch_r['params']['omega']; alpha_g=garch_r['params']['alpha[1]']; beta_g=garch_r['params']['beta[1]']
sigma2_g=omega_g/(1-alpha_g-beta_g)
nic_g=np.sqrt(omega_g+alpha_g*z**2*sigma2_g+beta_g*sigma2_g)
jp=gjr_r['params']
omega_j=jp['omega']; alpha_j=jp['alpha[1]']; gamma_j=jp.get('gamma[1]',0); beta_j=jp['beta[1]']
sigma2_j=omega_j/(1-alpha_j-0.5*gamma_j-beta_j)
nic_gjr=np.sqrt(omega_j+(alpha_j+gamma_j*(z<0).astype(float))*z**2*sigma2_j+beta_j*sigma2_j)
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(z,nic_g,color='#185FA5',linewidth=2.2,label='GARCH(1,1) — symmetric')
ax.plot(z,nic_gjr,color='#E24B4A',linewidth=2.2,label='GJR-GARCH — asymmetric')
ax.axvline(0,color='gray',linewidth=0.8,linestyle='--')
ax.fill_between(z[z<0],nic_gjr[z<0],nic_g[z<0],alpha=0.15,color='#E24B4A',label='Leverage premium')
ax.set_title('News Impact Curves: GARCH vs GJR-GARCH')
ax.set_xlabel('Standardised Innovation (z)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/03_news_impact_curves.png',dpi=150,bbox_inches='tight')
plt.show()